# 05 - Data Enrichment and Hypothesis Testing
This notebook applies all trained models to the clean dataset to output the final enriched dataset, and tests key research hypotheses (H1-H5).

In [1]:
import pandas as pd
import numpy as np
import os
import sys
from scipy.stats import chi2_contingency

sys.path.append("../src")
from config import (
    CLEAN_DATA_PATH, ENRICHED_DATA_PATH,
    HARM_INDIVIDUAL_MODEL_PATH, HARM_SOCIETAL_MODEL_PATH, AFFECTED_PARTY_MODEL_PATH
)
from models.harm_classifier import HarmClassifier
from models.affected_party import AffectedPartyClassifier

## Load data & models

In [2]:
df = pd.read_csv(CLEAN_DATA_PATH)
clf_ind = HarmClassifier.load(HARM_INDIVIDUAL_MODEL_PATH)
clf_soc = HarmClassifier.load(HARM_SOCIETAL_MODEL_PATH)
clf_ap = AffectedPartyClassifier.load(AFFECTED_PARTY_MODEL_PATH)

## Predict Missing Values & Enrich Dataset

In [3]:
print("Predicting Harm_Individual missing values...")
pred_ind = clf_ind.predict(df)
df['Harm_Individual_ML'] = pred_ind
df['Harm_Individual_IsPredicted'] = df['Harm_Individual'].isna()
df['Harm_Individual_Final'] = df['Harm_Individual'].fillna(df['Harm_Individual_ML'])

print("Predicting Harm_Societal missing values...")
pred_soc = clf_soc.predict(df)
df['Harm_Societal_ML'] = pred_soc
df['Harm_Societal_IsPredicted'] = df['Harm_Societal'].isna()
df['Harm_Societal_Final'] = df['Harm_Societal'].fillna(df['Harm_Societal_ML'])

print("Predicting Affected Party for all rows...")
df['AffectedParty_Final'] = clf_ap.predict(df)

df.to_csv(ENRICHED_DATA_PATH, index=False)
print(f"Enriched dataset saved to: {ENRICHED_DATA_PATH}")

Predicting Harm_Individual missing values...
Predicting Harm_Societal missing values...
Predicting Affected Party for all rows...
Enriched dataset saved to: /Users/berat/Desktop/ai ethics project antigravity/data/processed/aiaaic_enriched.csv


## Hypothesis 1: Algorithmic Injustice (Sector vs Affected Party)

In [4]:
# Explode sectors and affected parties
df_exp = df.copy()
df_exp['Sector_List'] = df_exp['Sector'].dropna().apply(lambda x: [s.strip() for s in x.split(';')])
df_exp = df_exp.explode('Sector_List')
df_exp['AP_List'] = df_exp['AffectedParty_Final'].dropna().apply(lambda x: [a.strip() for a in x.split(';')])
df_exp = df_exp.explode('AP_List')
df_exp = df_exp.reset_index(drop=True)

contingency = pd.crosstab(df_exp['Sector_List'], df_exp['AP_List'])
chi2, p, dof, expected = chi2_contingency(contingency)
print(f"Chi-square Test of Sector vs Affected Party:")
print(f"  Chi2 Stat: {chi2:.4f}")
print(f"  p-value: {p:.4g}")
if p < 0.05:
    print("Result: Statistically significant. Certain sectors systematically harm specific groups.")
else:
    print("Result: Not statistically significant.")

Chi-square Test of Sector vs Affected Party:
  Chi2 Stat: 6269.1144
  p-value: 0
Result: Statistically significant. Certain sectors systematically harm specific groups.


## Hypothesis 2: Repeat Offenders (Developer Concentration)

In [5]:
all_devs = df['Developer'].dropna().apply(lambda x: [d.strip() for d in x.split(';')]).explode()
counts = all_devs.value_counts()
pct = (counts / len(df)) * 100
print("Top 10 Developer Concentration:")
for i, (dev, count) in enumerate(counts.head(10).items()):
    print(f"  {i+1}. {dev}: {count} incidents ({pct.iloc[i]:.2f}% of total)")

Top 10 Developer Concentration:
  1. OpenAI: 236 incidents (10.50% of total)
  2. Google: 142 incidents (6.32% of total)
  3. Meta: 111 incidents (4.94% of total)
  4. Amazon: 74 incidents (3.29% of total)
  5. Tesla: 73 incidents (3.25% of total)
  6. Microsoft: 70 incidents (3.12% of total)
  7. xAI: 49 incidents (2.18% of total)
  8. ByteDance/TikTok: 26 incidents (1.16% of total)
  9. Apple: 25 incidents (1.11% of total)
  10. Anthropic: 24 incidents (1.07% of total)
